In [1]:
import pandas as pd
import os

if os.path.exists('/root/Public_Storage/madelab_khw/lg_aimers/dataset/train.csv'):
    df = pd.read_csv('/root/Public_Storage/madelab_khw/lg_aimers/dataset/train.csv')
else:
    print('none')

os.system('nvidia-smi')

Tue Feb 25 12:34:40 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.01             Driver Version: 535.183.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A5000               Off | 00000000:01:00.0 Off |                  Off |
| 30%   28C    P8              16W / 230W |      9MiB / 24564MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

0

In [2]:
import sys
print(sys.executable)


/root/anaconda3/envs/khw/bin/python


In [3]:
import torch
if torch.cuda.is_available() :
    # torch.cuda.device # Context-manager that changes the selected device.
    device = torch.device('cuda:1')
    print('GPU is available.')
else :
    device = torch.device('cpu')
    print('GPU is not available.')

print(device)

GPU is available.
cuda:1


In [4]:

def pre_process(df):
    threshold = 0.8
    df = df.loc[:, df.isnull().mean()<0.8]
    df = df.drop(columns=['ID'])
    df_sol = df['임신 성공 여부']
    df = df.drop(columns=['임신 성공 여부'])
    
    for col in df.columns:
        if df[col].dtype == 'object':  # 문자열 컬럼 처리
            df[col].fillna('Unknown', inplace=True)
        else:
            unique_count = df[col].nunique()
            if unique_count <= 4:
                mode = df[col].mode()[0]
                df[col].fillna(mode, inplace=True)
            else:
                df[col].fillna(df[col].mean(), inplace=True)
    
    
    return df, df_sol
        

In [5]:
pre_df, sol = pre_process(df)
print(pre_df.shape)
print(pre_df.dtypes)

(256351, 61)
시술 시기 코드        object
시술 당시 나이        object
시술 유형           object
특정 시술 유형        object
배란 자극 여부         int64
                ...   
기증 배아 사용 여부    float64
대리모 여부         float64
난자 채취 경과일      float64
난자 혼합 경과일      float64
배아 이식 경과일      float64
Length: 61, dtype: object


In [6]:
import pandas as pd

pd.set_option('display.max_rows', None)  # 모든 행 출력
pd.set_option('display.max_columns', None)  # 모든 열 출력
pd.set_option('display.expand_frame_repr', False)  # 가로 생략 방지


In [7]:
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

def pre_tf(pre_df, sol, scaling_method='standard'):
    category_columns = pre_df.select_dtypes(include=['object']).columns.tolist()
    numeric_columns = pre_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

    label_encoder = {}
    for col in category_columns:
        encoder = LabelEncoder()
        
        # ✅ NaN 값이 있다면 처리
        pre_df[col] = pre_df[col].astype(str).fillna("missing")  
        
        # ✅ LabelEncoder 적용 후 int64 변환
        pre_df[col] = encoder.fit_transform(pre_df[col]).astype(np.int64)  
        label_encoder[col] = encoder

    print("▶ Label Encoding 적용 후 데이터 타입 확인:")
    print(pre_df.dtypes)  # ✅ 여기서 category_columns가 int64로 유지되는지 확인

    # ✅ 숫자형 데이터만 스케일링 적용
    if scaling_method == 'minmax':
        scaler = MinMaxScaler()
    else:
        scaler = StandardScaler()

    pre_df[numeric_columns] = scaler.fit_transform(pre_df[numeric_columns])

    print("▶ 스케일링 적용 후 데이터 타입 확인:")
    print(pre_df.dtypes)  # ✅ 여기서 numeric_columns만 float64인지 확인

    X = pre_df[category_columns + numeric_columns]  
    y = sol  

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=pre_df['시술 당시 나이'])
    
    return X_train, X_valid, y_train, y_valid


In [8]:
X_train, X_valid, y_train, y_valid = pre_tf(pre_df, sol)
category_columns = pre_df.select_dtypes(include=['int64']).columns.tolist()
numeric_columns = pre_df.select_dtypes(include=['float64']).columns.tolist()

▶ Label Encoding 적용 후 데이터 타입 확인:
시술 시기 코드                int64
시술 당시 나이                int64
시술 유형                   int64
특정 시술 유형                int64
배란 자극 여부                int64
배란 유도 유형                int64
단일 배아 이식 여부           float64
착상 전 유전 진단 사용 여부      float64
남성 주 불임 원인              int64
남성 부 불임 원인              int64
여성 주 불임 원인              int64
여성 부 불임 원인              int64
부부 주 불임 원인              int64
부부 부 불임 원인              int64
불명확 불임 원인               int64
불임 원인 - 난관 질환           int64
불임 원인 - 남성 요인           int64
불임 원인 - 배란 장애           int64
불임 원인 - 여성 요인           int64
불임 원인 - 자궁경부 문제         int64
불임 원인 - 자궁내막증           int64
불임 원인 - 정자 농도           int64
불임 원인 - 정자 면역학적 요인      int64
불임 원인 - 정자 운동성          int64
불임 원인 - 정자 형태           int64
배아 생성 주요 이유             int64
총 시술 횟수                 int64
클리닉 내 총 시술 횟수           int64
IVF 시술 횟수               int64
DI 시술 횟수                int64
총 임신 횟수                 int64
IVF 임신 횟수               int64
DI 임신 횟

In [9]:
import pandas as pd
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
import numpy as np

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_valid_tensor = torch.tensor(X_valid.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_valid_tensor = torch.tensor(y_valid.values, dtype=torch.long)

print(X_train_tensor.shape)


torch.Size([205080, 61])


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import rtdl
from sklearn.utils.class_weight import compute_class_weight

# ✅ 1️⃣ GPU 캐시 메모리 정리 (매 Epoch 실행)
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
torch.set_float32_matmul_precision('high')  # 🚀 메모리 최적화

# ✅ 2️⃣ 수치형 데이터 선택 + float16 변환
numeric_columns = pre_df.select_dtypes(include=['float64']).columns.tolist()
print(len(numeric_columns),'numeric_columns')
X_train_num = torch.tensor(X_train[numeric_columns].values, dtype=torch.float16).to(device)
X_valid_num = torch.tensor(X_valid[numeric_columns].values, dtype=torch.float16).to(device)

category_columns = pre_df.select_dtypes(include=['int64']).columns.tolist()
print(len(category_columns),'category_columns')

if category_columns:
    X_train_cat = torch.tensor(X_train[category_columns].values, dtype=torch.int64).to(device)
    X_valid_cat = torch.tensor(X_valid[category_columns].values, dtype=torch.int64).to(device)
else:
    print('none')

print('pre_df', pre_df.dtypes)
print()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ 3️⃣ FT-Transformer 모델 생성 (모델 크기 추가 축소)
model = rtdl.FTTransformer.make_baseline(
    n_num_features=len(numeric_columns),
    cat_cardinalities=[len(df[col].unique()) for col in category_columns] if category_columns else [],
    last_layer_query_idx=[-1],
    d_out=1,

    # 🚀 모델 크기 절반 추가 축소 (메모리 절약)
    d_token=32,   # ✅ 64 → 32
    n_blocks=1,   # ✅ 2 → 1
    attention_dropout=0.05,  
    ffn_d_hidden=64,  
    ffn_dropout=0.1,  
    residual_dropout=0.025  
)

if torch.cuda.device_count() > 1:
    print(f"🔥 Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

model.to(device)

# ✅ 4️⃣ 클래스 불균형 보정 (pos_weight 추가)
class_weights = compute_class_weight(class_weight='balanced', classes=[0,1], y=y_train)
pos_weight = torch.tensor([class_weights[1] / class_weights[0]], dtype=torch.float32).to(device)

# ✅ 5️⃣ Focal Loss 적용 (불균형 데이터 대응)
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        BCE_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss
        return loss.mean()

loss_fn = FocalLoss(alpha=0.5, gamma=2.0)

# ✅ 6️⃣ 옵티마이저 설정 (튜닝 적용)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# ✅ 7️⃣ 학습 실행 (🚀 Batch Size 추가 축소)
batch_size = 32  # ✅ 64 → 32 (메모리 절약)
best_auc = 0.0
patience = 20  
counter = 0

epochs = 500  
for epoch in range(epochs):
    optimizer.zero_grad()

    # ✅ Gradient Checkpointing 적용 (메모리 절약)
    with torch.cuda.amp.autocast():  
        output = model(X_train_num, X_train_cat) if category_columns else model(X_train_num, None)
        loss = loss_fn(output.squeeze(), y_train_tensor.to(device).float())  

    loss.backward()
    optimizer.step()
    
    # ✅ AUC 평가
    with torch.no_grad():
        y_pred_proba = torch.sigmoid(model(X_valid_num, X_valid_cat) if category_columns else model(X_valid_num, None)).cpu().numpy().squeeze()
        auc = roc_auc_score(y_valid, y_pred_proba)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f} - AUC: {auc:.4f}")
    
    

41 numeric_columns
20 category_columns
pre_df 시술 시기 코드                int64
시술 당시 나이                int64
시술 유형                   int64
특정 시술 유형                int64
배란 자극 여부              float64
배란 유도 유형                int64
단일 배아 이식 여부           float64
착상 전 유전 진단 사용 여부      float64
남성 주 불임 원인            float64
남성 부 불임 원인            float64
여성 주 불임 원인            float64
여성 부 불임 원인            float64
부부 주 불임 원인            float64
부부 부 불임 원인            float64
불명확 불임 원인             float64
불임 원인 - 난관 질환         float64
불임 원인 - 남성 요인         float64
불임 원인 - 배란 장애         float64
불임 원인 - 여성 요인         float64
불임 원인 - 자궁경부 문제       float64
불임 원인 - 자궁내막증         float64
불임 원인 - 정자 농도         float64
불임 원인 - 정자 면역학적 요인    float64
불임 원인 - 정자 운동성        float64
불임 원인 - 정자 형태         float64
배아 생성 주요 이유             int64
총 시술 횟수                 int64
클리닉 내 총 시술 횟수           int64
IVF 시술 횟수               int64
DI 시술 횟수                int64
총 임신 횟수                 int64
IVF 임신 횟수               

In [11]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score

# ✅ (1) 모델 평가 모드 설정
model.eval()

# ✅ (2) 검증 데이터 예측
with torch.no_grad():
    X_valid_num = X_valid_num.to(device)
    print(type(X_valid_cat))
    X_valid_cat = X_valid_cat.to(device) 
    
    output = model(X_valid_num, X_valid_cat)
    
    # 🚀 Sigmoid 적용 후 NumPy 변환
    y_pred_proba = torch.sigmoid(output).cpu().numpy().squeeze()

# 🚀 0.5 이상이면 1, 아니면 0으로 변환
y_pred = (y_pred_proba >= 0.5).astype(int)

# ✅ NaN/Inf 값 체크
assert not (torch.isnan(output).any() or torch.isinf(output).any()), "🔥 예측값에 NaN 또는 Inf 존재!"

# ✅ (3) 평가 지표 계산
acc = accuracy_score(y_valid, y_pred)
f1 = f1_score(y_valid, y_pred, average='macro')  
recall = recall_score(y_valid, y_pred, average='macro')
precision = precision_score(y_valid, y_pred, average='macro')
roc_auc = roc_auc_score(y_valid, y_pred_proba)

# ✅ (4) 결과 출력
print(f"✅ FT-Transformer Accuracy: {acc:.4f}")
print(f"🚀 FT-Transformer F1 Score: {f1:.4f}")
print(f"🚀 FT-Transformer Recall: {recall:.4f}")
print(f"🚀 FT-Transformer Precision: {precision:.4f}")
print(f"🚀 FT-Transformer ROC-AUC: {roc_auc:.4f}")

# ✅ 추가 디버깅 (확률값 체크)
print(f"Min: {y_pred_proba.min()}, Max: {y_pred_proba.max()}, Mean: {y_pred_proba.mean()}")


<class 'torch.Tensor'>
✅ FT-Transformer Accuracy: 0.7390
🚀 FT-Transformer F1 Score: 0.4249
🚀 FT-Transformer Recall: 0.5000
🚀 FT-Transformer Precision: 0.3695
🚀 FT-Transformer ROC-AUC: 0.5500
Min: 0.27123042941093445, Max: 0.44751691818237305, Mean: 0.4200785160064697


/root/anaconda3/envs/khw/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
